# Third-Party Vendor Governance: Vendor Risk Scoring## InsureLogic AI: Vendor Risk Assessment & SLA Compliance DashboardThis notebook implements vendor risk scoring and SLA compliance monitoring for InsureLogic AI's 5 AI vendor relationships.---### Quick Reference: Key Concepts**Vendor Risk Scoring Criteria**:- **Security Posture** (30%): Certifications (SOC2, ISO27001), encryption, access controls- **Data Handling** (25%): Data residency, retention policies, processing agreements- **Transparency** (20%): Model explainability, bias testing, documentation- **Financial Stability** (15%): Revenue, market position, business continuity- **SLA Quality** (10%): Uptime guarantees, response times, penalty structures**Risk Score Scale**: 1 (Lowest Risk) → 5 (Highest Risk)- 1.0-1.5: Highly Recommended — Strong across all dimensions- 1.5-2.5: Recommended — Good overall with minor gaps- 2.5-3.5: Acceptable with Monitoring — Notable gaps requiring attention- 3.5-5.0: Not Recommended — Significant governance concerns**Vendor Governance Best Practices**:- Maintain 2+ active vendor relationships for redundancy- Review vendor risk quarterly; SLA compliance monthly- Test failover procedures at least annually- Require 7+ days advance notice for model updates### ScenarioInsureLogic AI uses GenAI for claims processing, fraud detection, and customer chatbots across 5 vendors. After a vendor model update caused false claim denials, the CRO wants formal risk scoring and SLA monitoring.

## Step 1: Setup and Vendor Data

In [ ]:
# ============================================================# VENDOR EVALUATION SCORES (0-100 per criterion)# ============================================================vendors_df = pd.DataFrame({    'Vendor': ['NovaMind API', 'CloudVault AI', 'TitanScale AI', 'DeepLens AI', 'SafeGuard AI'],    'Security_Score': [85, 95, 95, 85, 85],    'Data_Handling_Score': [75, 90, 85, 70, 85],    'Model_Transparency': [90, 70, 70, 60, 90],    'Financial_Stability': [95, 100, 100, 95, 85],    'SLA_Compliance': [85, 90, 95, 85, 85],    'Exit_Strategy': [90, 95, 90, 70, 95],    'AI_Model_Governance': [None, None, None, None, None]  # TODO: Assess these in your Excel evaluation})

## Step 2: Define Risk Scoring WeightsVendor risk scoring requires weighting criteria by importance. This is a governance decision — the weights reflect what matters most for InsureLogic's insurance context.**Your task**: Define the scoring weights dictionary. Consider that for an insurance company:- Security is paramount (customer financial data)- Data handling matters for regulatory compliance- Transparency is critical for claims explainability- Financial stability ensures vendor longevity- SLA quality affects operational reliability- AI Model Governance is critical for GenAI vendors (training data provenance, model update notification, fine-tuning portability, liability terms)

In [ ]:
# TODO: Define scoring weights that sum to 1.0# Consider which criteria matter most for an insurance company processing claimsscoring_weights = {    'Security_Score': 0.0,       # TODO: assign weight (suggested range: 0.20-0.35)    'Data_Handling_Score': 0.0,  # TODO    'Model_Transparency': 0.0,   # TODO    'Financial_Stability': 0.0,  # TODO    'SLA_Compliance': 0.0,       # TODO    'Exit_Strategy': 0.0,        # TODO    'AI_Model_Governance': 0.0   # TODO: New 7th criterion (suggested: 0.08-0.12)}# Verify weights sum to 1.0total = sum(scoring_weights.values())print(f'Total weight: {total:.2f} (must equal 1.00)')assert abs(total - 1.0) < 0.01, f'Weights must sum to 1.0, got {total}'

## Step 3: Calculate Risk ScoresConvert vendor evaluation scores into a single risk score (1-5 scale) using your weights.**Your task**: Implement the weighted score calculation. The logic:1. For each vendor, compute a weighted average of their scores (0-100 scale)2. Convert to risk scale: 100 → risk 1.0 (lowest), 0 → risk 5.0 (highest)   Formula: `risk = 5 - (weighted_score / 100) * 4`

In [ ]:
def calculate_risk_scores(vendor_data, weights):
    """Calculate weighted risk scores for each vendor. Returns list of scores (1-5)."""
    risk_scores = []
    for _, row in vendor_data.iterrows():
        # TODO: Calculate weighted_score by summing (row[metric] * weight) for each metric
        weighted_score = 0  # TODO: replace with actual calculation
        
        # TODO: Convert to risk scale (1-5) using: risk = 5 - (weighted_score / 100) * 4
        risk_level = 0  # TODO: replace with actual conversion
        
        risk_scores.append(round(risk_level, 2))
    return risk_scores

vendors_df['Risk_Score'] = calculate_risk_scores(vendors_df, scoring_weights)
print('Vendor Risk Scores:')
print(vendors_df[['Vendor', 'Risk_Score']].to_string(index=False))

## Step 4: Vendor Recommendation LogicBased on risk scores and SLA compliance, assign a recommendation category for each vendor.**Your task**: Implement the recommendation function using these governance thresholds:- Risk ≤ 1.5 AND SLA compliant → "Highly Recommended"- Risk ≤ 2.5 AND SLA compliant → "Recommended"- Risk ≤ 3.5 → "Acceptable (with monitoring)"- Risk > 3.5 → "Not Recommended"

In [ ]:
# First, calculate SLA compliance (pre-provided)
uptime_threshold = 99.9
latency_threshold = 800

compliance_df = sla_df.groupby('Vendor').agg(
    Avg_Uptime=('Uptime', 'mean'),
    Avg_Latency=('Latency_ms', 'mean')
).round(3).reset_index()

compliance_df['SLA_Compliant'] = (
    (compliance_df['Avg_Uptime'] >= uptime_threshold) & 
    (compliance_df['Avg_Latency'] <= latency_threshold)
)

# Merge risk scores with compliance
summary = vendors_df[['Vendor', 'Risk_Score']].merge(compliance_df, on='Vendor')

def get_recommendation(risk_score, sla_compliant):
    """
    TODO: Return recommendation string based on risk score and SLA compliance.
    Use the thresholds described above.
    """
    # TODO: Implement recommendation logic
    return 'TODO'

summary['Recommendation'] = summary.apply(
    lambda row: get_recommendation(row['Risk_Score'], row['SLA_Compliant']), axis=1
)

print('Vendor Assessment Summary:')
print(summary.to_string(index=False))

## Step 5: Risk Comparison Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sorted_v = vendors_df.sort_values('Risk_Score')
colors = ['#2ecc71' if s <= 1.5 else '#f39c12' if s <= 3 else '#e74c3c' for s in sorted_v['Risk_Score']]
bars = ax.barh(sorted_v['Vendor'], sorted_v['Risk_Score'], color=colors)
ax.set_xlabel('Risk Score (1=Lowest, 5=Highest)', fontsize=12, fontweight='bold')
ax.set_title('Vendor Risk Assessment — InsureLogic AI', fontsize=14, fontweight='bold')
ax.set_xlim(0, 5)
ax.axvline(x=2.5, color='gray', linestyle='--', alpha=0.5, label='Acceptable Threshold')
ax.legend()
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.1, bar.get_y() + bar.get_height()/2, f'{w:.2f}', ha='left', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('vendor_risk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: SLA Compliance Dashboard

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

for vendor in vendors_df['Vendor']:
    vd = sla_df[sla_df['Vendor'] == vendor].sort_values('Day')
    ax1.plot(vd['Day'], vd['Uptime'], marker='o', label=vendor, linewidth=1.5, markersize=3)
ax1.axhline(y=uptime_threshold, color='red', linestyle='--', linewidth=2, label=f'SLA Threshold ({uptime_threshold}%)')
ax1.set_ylabel('Uptime (%)', fontweight='bold')
ax1.set_title('30-Day Uptime Tracking', fontsize=13, fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.set_ylim(98, 100.2)
ax1.grid(True, alpha=0.3)

for vendor in vendors_df['Vendor']:
    vd = sla_df[sla_df['Vendor'] == vendor].sort_values('Day')
    ax2.plot(vd['Day'], vd['Latency_ms'], marker='o', label=vendor, linewidth=1.5, markersize=3)
ax2.axhline(y=latency_threshold, color='red', linestyle='--', linewidth=2, label=f'SLA Threshold ({latency_threshold}ms)')
ax2.set_xlabel('Day', fontweight='bold')
ax2.set_ylabel('Latency (ms)', fontweight='bold')
ax2.set_title('30-Day Latency Tracking', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sla_compliance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Radar Chart — Multi-Dimensional Comparison

In [ ]:
top3 = vendors_df.nsmallest(3, 'Risk_Score')
categories = ['Security', 'Data Handling', 'Transparency', 'Financial', 'SLA', 'Exit Strategy']
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(projection='polar'))
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, (_, row) in enumerate(top3.iterrows()):
    ax = axes[idx]
    values = [row['Security_Score'], row['Data_Handling_Score'], row['Model_Transparency'],
              row['Financial_Stability'], row['SLA_Compliance'], row['Exit_Strategy']]
    values += values[:1]
    angles = [n / len(categories) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=colors[idx])
    ax.fill(angles, values, alpha=0.25, color=colors[idx])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_title(f"{row['Vendor']}\n(Risk: {row['Risk_Score']})", fontsize=11, fontweight='bold', pad=20)

plt.suptitle('Multi-Dimensional Vendor Comparison (Top 3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('vendor_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4.5: Scenario Analysis — Model Update IncidentThe InsureLogic incident mentioned in the scenario involved an unannounced vendor model update that caused false claim denials.**TODO**: Which vendor(s) in your assessment have the highest risk around model update notification and change management? How would you adjust their scores to reflect this risk?

## Step 8: Final Assessment Report

In [ ]:
print('='*90)
print('VENDOR ASSESSMENT REPORT — InsureLogic AI Claims Processing System')
print('='*90)

for _, row in summary.sort_values('Risk_Score').iterrows():
    print(f"\n  {row['Vendor']}")
    print(f"    Risk Score: {row['Risk_Score']:.2f}/5.0")
    print(f"    Avg Uptime: {row['Avg_Uptime']:.3f}% {'✓' if row['Avg_Uptime'] >= uptime_threshold else '✗'}")
    print(f"    Avg Latency: {row['Avg_Latency']:.1f}ms {'✓' if row['Avg_Latency'] <= latency_threshold else '✗'}")
    print(f"    Recommendation: {row['Recommendation']}")

print('\n' + '='*90)
print(f"Scoring weights used: {scoring_weights}")
print(f"SLA thresholds: Uptime >= {uptime_threshold}%, Latency <= {latency_threshold}ms")

## Step 9: Exit Strategy Analysis — Vendor Lock-inFor GenAI vendors, "exit strategy" extends beyond simple data portability.**TODO**: Review each vendor's exit strategy score. Which vendor(s) would be most expensive to migrate away from if their prices increased significantly or their service degraded?

## Step 10: Sensitivity Analysis (Stretch Goal)Small changes in vendor scores can flip recommendation tiers.**TODO** (Optional): Pick one vendor and adjust all their criterion scores by ±10 points. How do their recommendations change? Which criteria matter most for your final decision?